In [46]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
%matplotlib inline

print("Libraries imported successfully!")

Libraries imported successfully!


In [79]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(repr(os.path.join(root, f)))

'/kaggle/input/datasets/shubhangi005/tstimg9/Screenshot 2026-07-10 131710.png'
'/kaggle/input/datasets/shubhangi005/tstimg9/Screenshot 2026-07-10 131759.png'


In [81]:
# Hardcoded paths for your two target images
valid_images = [
    '/kaggle/input/datasets/shubhangi005/tstimg9/Screenshot 2026-07-10 131759.png',
    '/kaggle/input/datasets/shubhangi005/tstimg9/Screenshot 2026-07-10 131710.png'

]

# Verify both paths exist
existing_images = [img for img in valid_images if os.path.exists(img)]

if len(existing_images) < len(valid_images):
    print("Warning: One or both image paths could not be found. Please double-check the dataset attachment.")
else:
    print(f"Success! Both images found and loaded.")
    
    # Load the first image as the calibration reference for the sliders
    ref_img = cv2.imread(existing_images[0])
    ref_img = cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB)
    H, W, _ = ref_img.shape
    print(f"Reference Image Dimensions: Width={W}, Height={H}")

Success! Both images found and loaded.
Reference Image Dimensions: Width=982, Height=487


In [82]:
import ipywidgets as widgets

# Define initial layout positions for the 4 corners
init_p1 = [int(W * 0.2), int(H * 0.2)]  # Top-Left
init_p2 = [int(W * 0.8), int(H * 0.2)]  # Top-Right
init_p3 = [int(W * 0.8), int(H * 0.8)]  # Bottom-Right
init_p4 = [int(W * 0.2), int(H * 0.8)]  # Bottom-Left

# Create separate sliders for X and Y for each of the 4 points
p1_x = widgets.IntSlider(value=init_p1[0], min=0, max=W, description='P1 X (TL):')
p1_y = widgets.IntSlider(value=init_p1[1], min=0, max=H, description='P1 Y (TL):')

p2_x = widgets.IntSlider(value=init_p2[0], min=0, max=W, description='P2 X (TR):')
p2_y = widgets.IntSlider(value=init_p2[1], min=0, max=H, description='P2 Y (TR):')

p3_x = widgets.IntSlider(value=init_p3[0], min=0, max=W, description='P3 X (BR):')
p3_y = widgets.IntSlider(value=init_p3[1], min=0, max=H, description='P3 Y (BR):')

p4_x = widgets.IntSlider(value=init_p4[0], min=0, max=W, description='P4 X (BL):')
p4_y = widgets.IntSlider(value=init_p4[1], min=0, max=H, description='P4 Y (BL):')

# Global dictionary to hold current point values
saved_points = {}

def update_points_roi(p1x, p1y, p2x, p2y, p3x, p3y, p4x, p4y):
    global saved_points
    saved_points = {
        'p1': [p1x, p1y],
        'p2': [p2x, p2y],
        'p3': [p3x, p3y],
        'p4': [p4x, p4y]
    }
    
    # Render preview with lines connecting the 4 points
    preview = ref_img.copy()
    pts = np.array([saved_points['p1'], saved_points['p2'], saved_points['p3'], saved_points['p4']], np.int32)
    pts = pts.reshape((-1, 1, 2))
    
    # Draw polygon boundary and label points
    cv2.polylines(preview, [pts], isClosed=True, color=(255, 0, 0), thickness=4)
    for i, pt in enumerate([saved_points['p1'], saved_points['p2'], saved_points['p3'], saved_points['p4']]):
        cv2.circle(preview, tuple(pt), 8, (0, 255, 0), -1)
        cv2.putText(preview, f"P{i+1}", (pt[0]+10, pt[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2)
        
    plt.figure(figsize=(12, 7))
    plt.imshow(preview)
    plt.title("Adjust Sliders to Position P1, P2, P3, P4")
    plt.axis('on')
    plt.show()

# Layout sliders nicely in pairs
ui = widgets.VBox([
    widgets.HBox([p1_x, p1_y]),
    widgets.HBox([p2_x, p2_y]),
    widgets.HBox([p3_x, p3_y]),
    widgets.HBox([p4_x, p4_y])
])

out = widgets.interactive_output(update_points_roi, {
    'p1x': p1_x, 'p1y': p1_y,
    'p2x': p2_x, 'p2y': p2_y,
    'p3x': p3_x, 'p3y': p3_y,
    'p4x': p4_x, 'p4y': p4_y
})

display(ui, out)










Output()

In [83]:
src_pts = np.float32([saved_points['p1'], saved_points['p2'], saved_points['p3'], saved_points['p4']])

width_top = np.linalg.norm(src_pts[1] - src_pts[0])
width_bottom = np.linalg.norm(src_pts[2] - src_pts[3])
max_width = max(int(width_top), int(width_bottom))

height_right = np.linalg.norm(src_pts[2] - src_pts[1])
height_left = np.linalg.norm(src_pts[3] - src_pts[0])
max_height = max(int(height_right), int(height_left))

dst_pts = np.float32([
    [0, 0],
    [max_width - 1, 0],
    [max_width - 1, max_height - 1],
    [0, max_height - 1]
])

matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)

print(f"ROI captured: src_pts={src_pts.tolist()}")
print(f"Flattened size: {max_width} x {max_height}")


   
                   
       

ROI captured: src_pts=[[385.0, 97.0], [567.0, 97.0], [847.0, 389.0], [496.0, 389.0]]
Flattened size: 351 x 404


In [84]:
landmark_x = widgets.IntSlider(value=int(W*0.85), min=0, max=W, description='Landmark X:')
landmark_y = widgets.IntSlider(value=int(H*0.85), min=0, max=H, description='Landmark Y:')
patch_half_size = 50  # slightly larger patch for the bigger, flatter signboard face

def preview_landmark(lx, ly):
    preview = ref_img.copy()
    cv2.rectangle(preview, (lx - patch_half_size, ly - patch_half_size),
                  (lx + patch_half_size, ly + patch_half_size), (255, 0, 0), 3)
    plt.figure(figsize=(12, 7))
    plt.imshow(preview)
    plt.title("Position box tightly on the signboard face (not the barrel)")
    plt.show()

out2 = widgets.interactive_output(preview_landmark, {'lx': landmark_x, 'ly': landmark_y})
display(widgets.VBox([landmark_x, landmark_y]), out2)

Output()

In [85]:
def find_landmark_transform(cur_bgr, template, lx, ly, search_margin=250,
                              scale_range=(0.85, 1.15), scale_steps=25):
    y0 = max(0, ly - patch_half_size - search_margin)
    y1 = min(cur_bgr.shape[0], ly + patch_half_size + search_margin)
    x0 = max(0, lx - patch_half_size - search_margin)
    x1 = min(cur_bgr.shape[1], lx + patch_half_size + search_margin)
    search_region = cur_bgr[y0:y1, x0:x1]

    best_val = -1
    best_x, best_y, best_scale = lx, ly, 1.0

    for s in np.linspace(scale_range[0], scale_range[1], scale_steps):
        resized_template = cv2.resize(template, None, fx=s, fy=s, interpolation=cv2.INTER_LINEAR)
        th, tw = resized_template.shape[:2]
        if th >= search_region.shape[0] or tw >= search_region.shape[1] or th < 5 or tw < 5:
            continue

        result = cv2.matchTemplate(search_region, resized_template, cv2.TM_CCOEFF_NORMED)
        _, max_val, _, max_loc = cv2.minMaxLoc(result)

        if max_val > best_val:
            best_val = max_val
            best_x = x0 + max_loc[0] + tw / 2
            best_y = y0 + max_loc[1] + th / 2
            best_scale = s

    dx = best_x - lx
    dy = best_y - ly
    return dx, dy, best_scale, best_val


# ---- THIS PART MUST RUN TOO: builds the template from your slider position ----
lx = landmark_x.value
ly = landmark_y.value
ref_bgr = cv2.imread(existing_images[0])
template = ref_bgr[ly - patch_half_size: ly + patch_half_size,
                    lx - patch_half_size: lx + patch_half_size]
print(f"Template built from landmark at ({lx}, {ly}), patch size {template.shape}")

# ---- Now test it on both images ----
for img_path in existing_images:
    cur_bgr = cv2.imread(img_path)
    if img_path == existing_images[0]:
        dx, dy, scale, confidence = 0, 0, 1.0, 1.0
    else:
        dx, dy, scale, confidence = find_landmark_transform(cur_bgr, template, lx, ly)
    print(f"{os.path.basename(img_path)}: shift=({dx:.1f}px, {dy:.1f}px), scale={scale:.3f}, confidence={confidence:.3f}")


Template built from landmark at (694, 59), patch size (100, 100, 3)
Screenshot 2026-07-10 131759.png: shift=(0.0px, 0.0px), scale=1.000, confidence=1.000
Screenshot 2026-07-10 131710.png: shift=(-8.5px, -7.5px), scale=1.012, confidence=0.705


In [86]:
 output_dir_coord = "/kaggle/working/coordinate_based_output"
os.makedirs(output_dir_coord, exist_ok=True)

CONFIDENCE_THRESHOLD = 0.6
anchor = np.array([lx, ly], dtype=np.float32)

for img_path in existing_images:
    cur_bgr = cv2.imread(img_path)

    if img_path == existing_images[0]:
        dx, dy, scale, confidence = 0, 0, 1.0, 1.0
        status = "reference"
    else:
        dx, dy, scale, confidence = find_landmark_transform(cur_bgr, template, lx, ly)
        if confidence < CONFIDENCE_THRESHOLD:
            status = f"landmark not found reliably (conf={confidence:.2f}) - using unshifted ROI"
            dx, dy, scale = 0, 0, 1.0
        else:
            status = f"shift=({dx:.1f},{dy:.1f})px, scale={scale:.3f} - ROI adjusted"

    print(f"{os.path.basename(img_path)}: {status}")

    new_anchor = anchor + np.array([dx, dy], dtype=np.float32)
    shifted_src_pts = (src_pts - anchor) * scale + new_anchor
    shifted_src_pts = shifted_src_pts.astype(np.float32)

    same_shape_matrix = cv2.getPerspectiveTransform(shifted_src_pts, dst_pts)
    same_road_crop = cv2.warpPerspective(cur_bgr, same_shape_matrix, (max_width, max_height))

    base_name = os.path.basename(img_path)
    cv2.imwrite(os.path.join(output_dir_coord, f"coord_crop_{base_name}"), same_road_crop)

print(f"\nDone. Outputs in {output_dir_coord}")

Screenshot 2026-07-10 131759.png: reference
Screenshot 2026-07-10 131710.png: shift=(-8.5,-7.5)px, scale=1.012 - ROI adjusted

Done. Outputs in /kaggle/working/coordinate_based_output


In [87]:
output_dir_overlay2 = "/kaggle/working/coordinate_based_overlay"
os.makedirs(output_dir_overlay2, exist_ok=True)

for idx, img_path in enumerate(existing_images):
    cur_bgr = cv2.imread(img_path)

    if img_path == existing_images[0]:
        dx, dy, scale = 0, 0, 1.0
    else:
        dx, dy, scale, confidence = find_landmark_transform(cur_bgr, template, lx, ly)
        if confidence < CONFIDENCE_THRESHOLD:
            dx, dy, scale = 0, 0, 1.0

    new_anchor = anchor + np.array([dx, dy], dtype=np.float32)
    shifted_src_pts = (src_pts - anchor) * scale + new_anchor
    pts_int = shifted_src_pts.astype(np.int32).reshape((-1, 1, 2))

    overlay = cur_bgr.copy()
    poly_color = (255, 0, 0) if idx == 1 else (0, 255, 0)  # BGR: blue for target, green for reference
    cv2.polylines(overlay, [pts_int], isClosed=True, color=poly_color, thickness=4)

    base_name = os.path.basename(img_path)
    cv2.imwrite(os.path.join(output_dir_overlay2, f"shifted_roi_{base_name}"), overlay)

print(f"Done. Check {output_dir_overlay2}")

Done. Check /kaggle/working/coordinate_based_overlay
